In [5]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [6]:
from tensorflow.keras.datasets import imdb

In [7]:
# Load IMDB dataset, which provides reviews as sequences of word indices
(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=10000)

# Get the word index mapping
word_index = imdb.get_word_index()
# Reverse mapping to get words from indices
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])

# Function to decode reviews back to text
# The indices are offset by 3 because 0, 1, and 2 are reserved for "padding," "start of sequence," and "unknown."
def decode_review(text_indices):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in text_indices])

# Decode all reviews
decoded_train_reviews = [decode_review(seq) for seq in train_data]
decoded_test_reviews = [decode_review(seq) for seq in test_data]

# Convert numerical labels (0/1) to string labels ('negative'/'positive')
# to be compatible with cell 35e64659's map operation
sentiment_map_reverse = {0: 'negative', 1: 'positive'}
string_train_labels = [sentiment_map_reverse[label] for label in train_labels]
string_test_labels = [sentiment_map_reverse[label] for label in test_labels]

# Combine decoded reviews and string labels into a single DataFrame
all_reviews = decoded_train_reviews + decoded_test_reviews
all_sentiments = string_train_labels + string_test_labels

df = pd.DataFrame({'review': all_reviews, 'sentiment': all_sentiments})

print("DataFrame 'df' created with raw reviews and sentiment labels.")
print(df.head())

DataFrame 'df' created with raw reviews and sentiment labels.
                                              review sentiment
0  ? this film was just brilliant casting locatio...  positive
1  ? big hair big boobs bad music and a giant saf...  negative
2  ? this has to be one of the worst films of the...  negative
3  ? the ? ? at storytelling the traditional sort...  positive
4  ? worst mistake of my life br br i picked this...  negative


In [8]:
np.unique(train_labels)

array([0, 1])

In [9]:
# =========================
# 3. FIX COLUMN NAMES
# =========================
df.columns = ['review', 'sentiment']

In [10]:
# =========================
# 4. CLEAN TEXT
# =========================
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)      # remove HTML
    text = re.sub(r'[^a-zA-Z ]', '', text) # remove symbols
    return text

df['review'] = df['review'].apply(clean_text)

In [11]:
# =========================
# 5. CONVERT LABELS
# =========================
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

print("\nSentiment Count:")
print(df['sentiment'].value_counts())


Sentiment Count:
sentiment
1    25000
0    25000
Name: count, dtype: int64


In [12]:
# =========================
# 6. TF-IDF VECTORIZATION
# =========================
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['review']).toarray()
y = df['sentiment']

In [13]:

# =========================
# 7. TRAIN TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [14]:
# =========================
# 8. BUILD MODEL
# =========================
model = Sequential()

model.add(Dense(128, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

c:\Users\rrbag\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │       640,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 648,449 (2.47 MB)

 Trainable params: 648,449 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
# =========================
# 11. FINAL ACCURACY
# =========================
loss, accuracy = model.evaluate(X_test, y_test)
print("\nFinal Accuracy:", accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8795 - loss: 1.0748

Final Accuracy: 0.8794999718666077
